In [ ]:
# ABOUTME: Interactive notebook teaching similarity search using image embeddings
# ABOUTME: Covers cosine similarity, nearest neighbors, and building a simple image search engine

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torchvision
from torchvision import models, transforms
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image
from ipywidgets import interact, widgets
import jscatter
import base64
from io import BytesIO

%matplotlib widget


def pil_to_data_uri(img, size=128, fmt='JPEG'):
    """Convert a PIL image to a base64 data URI for jscatter tooltip preview."""
    thumb = img.copy()
    thumb.thumbnail((size, size))
    buf = BytesIO()
    thumb.save(buf, format=fmt)
    b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    mime = 'image/jpeg' if fmt == 'JPEG' else 'image/png'
    return f"data:{mime};base64,{b64}"

# Finding Twins: Similarity Search in Embedding Space

**Scenario:** You upload a photo of your golden retriever to Google Photos. Within seconds, it finds every other photo of your golden retriever — from different angles, lighting, backgrounds. How?

The answer is **similarity search in embedding space**:
1. Convert every photo to an embedding vector (as we learned in Notebook 03)
2. When you upload a query photo, convert it to a vector too
3. Find the vectors closest to your query — those are the most similar photos

In this notebook, we'll build a simple image search engine from scratch using **cosine similarity**.

## Cosine Similarity: Measuring the Angle Between Arrows

Imagine two arrows pointing from the origin. If they point in the **same direction**, they're similar. If they're **perpendicular**, they're unrelated. If they point in **opposite directions**, they're opposites.

$$\text{cosine similarity}(\vec{a}, \vec{b}) = \frac{\vec{a} \cdot \vec{b}}{|\vec{a}| \times |\vec{b}|}$$

| Value | Meaning | Analogy |
|-------|---------|---------|
| **1.0** | Identical direction | Two photos of the same cat |
| **0.5–0.9** | Similar | Same breed, different pose |
| **~0.0** | Unrelated | A cat and a truck |
| **-1.0** | Opposite | (Rare in practice) |

Cosine similarity only cares about **direction**, not magnitude — so it works well even if embeddings have different scales.

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

N_IMAGES = 60           # Larger sample for richer search results
K_NEIGHBORS = 5         # How many nearest neighbors to show
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "./data"
THUMBNAIL_SIZE = 128

print(f"Device: {DEVICE}")
print(f"Will load {N_IMAGES} images, search for {K_NEIGHBORS} nearest neighbors")

In [ ]:
# ============================================================================
# LOAD DATA + COMPUTE EMBEDDINGS
# ============================================================================

np.random.seed(SEED)
torch.manual_seed(SEED)

# Load Oxford Pets
dataset = torchvision.datasets.OxfordIIITPet(
    root=DATA_DIR, split='trainval',
    target_types='binary-category',
    download=True
)

cat_indices = [i for i, (_, label) in enumerate(dataset) if label == 0]
dog_indices = [i for i, (_, label) in enumerate(dataset) if label == 1]

n_per_class = N_IMAGES // 2
sampled_indices = np.concatenate([
    np.random.choice(cat_indices, n_per_class, replace=False),
    np.random.choice(dog_indices, n_per_class, replace=False),
])
np.random.shuffle(sampled_indices)

images = []
labels = []
label_names = {0: "Cat", 1: "Dog"}

for idx in sampled_indices:
    img, label = dataset[idx]
    images.append(img)
    labels.append(label_names[label])

# Load ResNet-18 and extract embeddings
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet.eval()
resnet = resnet.to(DEVICE)

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

embeddings_list = []
hook = resnet.avgpool.register_forward_hook(
    lambda m, inp, out: embeddings_list.append(out.squeeze().detach().cpu().numpy())
)

print("Computing embeddings...")
with torch.no_grad():
    for img in images:
        tensor = preprocess(img).unsqueeze(0).to(DEVICE)
        _ = resnet(tensor)

hook.remove()
embeddings = np.array(embeddings_list)

# PCA for visualization
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

# Generate data URIs for jscatter tooltips
thumbnail_uris = [pil_to_data_uri(img, size=THUMBNAIL_SIZE) for img in images]

# Compute full similarity matrix
sim_matrix = cosine_similarity(embeddings)

print(f"Loaded {len(images)} images, computed {embeddings.shape[1]}-dim embeddings")
print(f"Similarity matrix shape: {sim_matrix.shape}")

## The Similarity Landscape

The heatmap below shows cosine similarity between every pair of images (sorted by label). Notice the **block diagonal** — images within the same class are more similar to each other.

In [ ]:
# ============================================================================
# SIMILARITY HEATMAP
# ============================================================================

sort_idx = np.argsort(labels)
sorted_labels = [labels[i] for i in sort_idx]
sorted_sim_matrix = sim_matrix[np.ix_(sort_idx, sort_idx)]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    sorted_sim_matrix,
    cmap='RdYlBu_r',
    center=0.5,
    square=True,
    cbar_kws={'label': 'Cosine Similarity'},
    ax=ax,
)

n_cats = sorted_labels.count('Cat')
ax.axhline(y=n_cats, color='black', linewidth=2, linestyle='--')
ax.axvline(x=n_cats, color='black', linewidth=2, linestyle='--')
ax.set_title('Cosine Similarity Matrix (sorted by class)', fontsize=14, fontweight='bold')
ax.set_xlabel('Cats ← | → Dogs')
ax.set_ylabel('Cats ← | → Dogs')

plt.tight_layout()
plt.show()

## Embedding Map

Explore the embedding space interactively. Hover to see images, zoom into clusters.

In [ ]:
# ============================================================================
# JSCATTER EMBEDDING MAP
# ============================================================================

df = pd.DataFrame({
    'pca_x': embeddings_2d[:, 0],
    'pca_y': embeddings_2d[:, 1],
    'label': pd.Categorical(labels),
    'thumbnail': thumbnail_uris,
    'index': list(range(len(images))),
})

scatter = jscatter.Scatter(
    data=df, x='pca_x', y='pca_y',
    height=500,
)
scatter.color(by='label', map={'Dog': '#e74c3c', 'Cat': '#3498db'})
scatter.size(8)
scatter.tooltip(
    enable=True,
    properties=['label', 'index'],
    preview='thumbnail',
    preview_type='image',
    preview_image_size='contain',
)
scatter.legend(True)
scatter.show()

## Building a Search Engine

Pick a query image from the dropdown. The search engine will find the K most similar images by computing cosine similarity against every other image in the dataset.

In [ ]:
# ============================================================================
# INTERACTIVE NEAREST NEIGHBOR SEARCH
# ============================================================================

fig_search, axes_search = plt.subplots(1, K_NEIGHBORS + 1, figsize=(3 * (K_NEIGHBORS + 1), 3.5))

query_options = {f"#{i} ({labels[i]})": i for i in range(len(images))}

@interact(
    query=widgets.Dropdown(
        options=query_options,
        value=0,
        description='Query image:',
        style={'description_width': 'initial'},
    ),
    k=widgets.IntSlider(
        min=1, max=10, value=K_NEIGHBORS,
        description='Neighbors:',
    ),
)
def search_neighbors(query, k):
    for ax in axes_search:
        ax.clear()
        ax.axis('off')

    # Get similarities to query (exclude self)
    sims = sim_matrix[query].copy()
    sims[query] = -1  # Exclude self
    neighbor_indices = np.argsort(sims)[::-1][:k]

    # Show query
    axes_search[0].imshow(images[query])
    axes_search[0].set_title(f'QUERY\n{labels[query]}', fontsize=11,
                              fontweight='bold', color='forestgreen')
    axes_search[0].axis('off')

    # Show neighbors
    for j, idx in enumerate(neighbor_indices):
        if j + 1 < len(axes_search):
            axes_search[j + 1].imshow(images[idx])
            match = '=' if labels[idx] == labels[query] else '\u2260'
            color = 'forestgreen' if labels[idx] == labels[query] else 'crimson'
            axes_search[j + 1].set_title(
                f'#{j+1} ({labels[idx]}) {match}\nsim={sims[idx]:.3f}',
                fontsize=10, fontweight='bold', color=color
            )
            axes_search[j + 1].axis('off')

    # Highlight neighbors in the jscatter plot
    scatter.selection(list(neighbor_indices) + [query])

    fig_search.suptitle('Nearest Neighbor Search', fontsize=14, fontweight='bold')
    fig_search.tight_layout()
    fig_search.canvas.draw_idle()

## When Does Similarity Search Fail?

No embedding is perfect. Let's find cases where the nearest neighbor belongs to a **different class** — these are the "confused" images that sit near the decision boundary.

In [ ]:
# ============================================================================
# FIND CONFUSED IMAGES
# ============================================================================

confused_pairs = []
for i in range(len(images)):
    sims = sim_matrix[i].copy()
    sims[i] = -1
    nearest = np.argmax(sims)
    if labels[nearest] != labels[i]:
        confused_pairs.append((i, nearest, sims[nearest]))

if confused_pairs:
    # Sort by similarity (highest = most confusing)
    confused_pairs.sort(key=lambda x: -x[2])

    n_show = min(3, len(confused_pairs))
    fig_confused, axes_confused = plt.subplots(n_show, 2, figsize=(6, 3 * n_show))
    if n_show == 1:
        axes_confused = axes_confused.reshape(1, -1)

    for row, (query_idx, neighbor_idx, sim) in enumerate(confused_pairs[:n_show]):
        axes_confused[row, 0].imshow(images[query_idx])
        axes_confused[row, 0].set_title(f'{labels[query_idx]} (query)', fontsize=11, fontweight='bold')
        axes_confused[row, 0].axis('off')

        axes_confused[row, 1].imshow(images[neighbor_idx])
        axes_confused[row, 1].set_title(
            f'{labels[neighbor_idx]} (nearest, sim={sim:.3f})',
            fontsize=11, fontweight='bold', color='crimson'
        )
        axes_confused[row, 1].axis('off')

    fig_confused.suptitle('Confused Pairs: Nearest Neighbor is Wrong Class',
                          fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f"Found {len(confused_pairs)} images whose nearest neighbor is the wrong class")
else:
    print("No confused images found — the embeddings perfectly separate dogs and cats!")

## Summary

- **Cosine similarity** measures the angle between embedding vectors (1.0 = identical, 0.0 = unrelated)
- **Nearest neighbor search**: find the K embeddings closest to your query — that's a search engine
- The **similarity matrix** reveals block-diagonal structure: same-class images cluster together
- **Edge cases** exist: some images sit near the decision boundary and confuse the search

This is exactly how Google Photos, Spotify recommendations, and semantic search engines work — just at a much larger scale with fancier embeddings.

**What's next?** In Notebook 06, we'll see something even more powerful: embedding **text and images in the same space**, so you can search for images by typing words.